# CNN para classificação dos escores de Gardner em blastocistos

Este notebook treina uma CNN multi-tarefa (multi-output) que recebe uma imagem estática de um blastocisto humano (dia 5 pós-inseminação) e prevê os três critérios do sistema de Gardner usados na seleção de embriões para transferência em FIV:

- **EXP** — grau de expansão/eclosão (ordinal, 5 classes: 0–4)
- **ICM** — qualidade da massa celular interna (ordinal, graus 0–2; classe 3 = não determinado)
- **TE** — qualidade do trofectoderma (ordinal, graus 0–2; classe 3 = não determinado)

A classe 3 do treino corresponde exatamente às imagens com EXP < 2, equivalente ao `ND` do teste gold (verificado na seção 2).

Dados: Kromp et al. (2023, *Sci Data*), em `data/images/`, `data/Gardner_train_silver.csv` (rótulos de um único especialista) e `data/Gardner_test_gold_onlyGardnerScores.csv` (rótulos de consenso de um consórcio — usado só como benchmark final).

## 1. Setup e imports

Requer `tensorflow` e `scikit-learn` instalados no `.venv` local (ver `requirements.txt`). Ambiente sem GPU — o treinamento roda em CPU.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "data"
IMAGES_DIR = os.path.join(DATA_DIR, "images")
TRAIN_CSV = os.path.join(DATA_DIR, "Gardner_train_silver.csv")
TEST_CSV = os.path.join(DATA_DIR, "Gardner_test_gold_onlyGardnerScores.csv")

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_HEIGHT, IMG_WIDTH = 168, 224  # mantém a proporção 4:3 da imagem original (512 de largura x 384 de altura)
BATCH_SIZE = 32

N_EXP, N_ICM, N_TE = 5, 4, 4

# Paleta consistente para os três alvos ao longo do notebook
TARGET_COLORS = {"EXP": "#4C72B0", "ICM": "#DD8452", "TE": "#55A868"}
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 2. Carga e exploração dos CSVs

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, sep=";")
train_df = train_df.rename(columns={"EXP_silver": "EXP", "ICM_silver": "ICM", "TE_silver": "TE"})

test_df = pd.read_csv(TEST_CSV, sep=";")
test_df = test_df.drop(columns=[c for c in test_df.columns if c.startswith("Unnamed")])
test_df = test_df.rename(columns={"EXP_gold": "EXP", "ICM_gold": "ICM", "TE_gold": "TE"})

# embryo_id = prefixo antes do "_" no nome do arquivo (ex: "0175_05.png" -> "0175")
train_df["embryo_id"] = train_df["Image"].str.extract(r"^(.+)_\d+\.png$")
test_df["embryo_id"] = test_df["Image"].str.extract(r"^(.+)_\d+\.png$")

print("Treino (silver):", train_df.shape, "| embriões únicos:", train_df["embryo_id"].nunique())
print("Teste (gold):", test_df.shape, "| embriões únicos:", test_df["embryo_id"].nunique())
display(train_df.head())
display(test_df.head())

In [ ]:
# ICM/TE só podem ser avaliados a partir de EXP >= 2.
# No silver, a classe 3 marca exatamente as imagens não avaliáveis; no gold, elas aparecem como "ND".
for target in ["ICM", "TE"]:
    assert ((train_df[target] == 3) == (train_df["EXP"] < 2)).all()
    assert ((test_df[target] == "ND") == (test_df["EXP"] < 2)).all()
    print(f"{target}: classe 3 no silver = {(train_df[target] == 3).sum()} imagens | ND no gold = {(test_df[target] == 'ND').sum()} imagens")

In [ ]:
# Distribuição de classes no treino (silver) por alvo
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, target in zip(axes, ["EXP", "ICM", "TE"]):
    counts = train_df[target].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=TARGET_COLORS[target])
    ax.set_title(f"{target} (treino silver)")
    ax.set_xlabel("classe")
    ax.set_ylabel("n amostras")
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
fig.suptitle("Distribuição de classes — desbalanceamento visível em EXP, ICM e TE")
fig.tight_layout()
plt.show()

In [ ]:
# Exemplos de imagens
from PIL import Image

sample_rows = train_df.sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(os.path.join(IMAGES_DIR, row["Image"]))
    ax.imshow(img)
    ax.set_title(f"EXP={row['EXP']} ICM={row['ICM']} TE={row['TE']}", fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 3. Split treino/validação por embrião

Usamos `GroupShuffleSplit` agrupando por `embryo_id` para garantir que imagens do mesmo embrião não apareçam simultaneamente em treino e validação (evita vazamento de informação).

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(gss.split(train_df, groups=train_df["embryo_id"]))

train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
val_split_df = train_df.iloc[val_idx].reset_index(drop=True)

assert set(train_split_df["embryo_id"]).isdisjoint(set(val_split_df["embryo_id"])), "Vazamento de embriões entre treino e validação!"

print(f"Treino: {len(train_split_df)} imagens / {train_split_df['embryo_id'].nunique()} embriões")
print(f"Validação: {len(val_split_df)} imagens / {val_split_df['embryo_id'].nunique()} embriões")

## 4. Pipelines `tf.data`

Carregamento/decodificação de PNG, redimensionamento para 224×168 (largura×altura) e normalização para [0,1]. Augmentação (flip horizontal/vertical, rotação ±15°, variação de brilho) é aplicada apenas no conjunto de treino, via camadas `keras.layers` ativas somente em modo de treinamento.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(15 / 360),
    layers.RandomBrightness(0.1, value_range=(0, 1)),
], name="augmentation")


def load_image(filename):
    path = tf.strings.join([IMAGES_DIR, filename], separator=os.sep)
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0
    return img


def make_dataset(df, training, batch_size=BATCH_SIZE):
    # y é uma tupla (EXP, ICM, TE), na mesma ordem das saídas do modelo —
    # passar um dict aqui quebra o Keras 3 quando combinado com sample_weight (ver make_dataset_with_weights)
    filenames = df["Image"].values
    labels = (
        df["EXP"].astype("int32").values,
        df["ICM"].astype("int32").values,
        df["TE"].astype("int32").values,
    )
    ds = tf.data.Dataset.from_tensor_slices((filenames, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    def _map(filename, label):
        img = load_image(filename)
        return img, label

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    if training:
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_split_df, training=True)
val_ds = make_dataset(val_split_df, training=False)

# smoke test: confirmar shapes de um lote
images_batch, labels_batch = next(iter(train_ds))
print("Batch de imagens:", images_batch.shape, images_batch.dtype)
for name, v in zip(["EXP", "ICM", "TE"], labels_batch):
    print(f"  {name}: {v.shape} {v.dtype}")

## 5. Definição do modelo: backbone compartilhado + 3 cabeças

In [ ]:
def conv_block(x, filters, n_convs=2):
    for _ in range(n_convs):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.MaxPooling2D()(x)
    return x


def build_model():
    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="image")
    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256, n_convs=1)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    exp_output = layers.Dense(N_EXP, activation="softmax", name="exp_output")(x)
    icm_output = layers.Dense(N_ICM, activation="softmax", name="icm_output")(x)
    te_output = layers.Dense(N_TE, activation="softmax", name="te_output")(x)

    return keras.Model(inputs=inputs, outputs=[exp_output, icm_output, te_output], name="gardner_cnn")


model = build_model()
model.summary()

## 6. Compilação: losses, loss_weights, métricas, otimizador

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    # listas na ordem das saídas do modelo (exp, icm, te) — ver nota em make_dataset sobre dicts + sample_weight
    loss=["sparse_categorical_crossentropy", "sparse_categorical_crossentropy", "sparse_categorical_crossentropy"],
    loss_weights=[1.0, 1.0, 1.0],
    metrics=[["accuracy"], ["accuracy"], ["accuracy"]],
)

## 7. Callbacks

In [ ]:
CHECKPOINT_PATH = os.path.join(MODELS_DIR, "gardner_cnn.keras")

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, start_from_epoch=15),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.CSVLogger(os.path.join(MODELS_DIR, "training_log.csv")),
]

## 8. Treinamento

Pesos de amostra por alvo (derivados de `compute_class_weight`) compensam o desbalanceamento de classes — o Keras multi-output não aceita um `class_weight` dict por saída diretamente, então convertemos para `sample_weight` por amostra em cada cabeça.

Os pesos são limitados a `MAX_CLASS_WEIGHT`, para que uma única imagem rara não domine o lote. Eles são usados só no treino. A validação não usa pesos: com apenas 4 imagens de ICM=2 na validação, cada uma pesaria dezenas de vezes mais que as demais, e a `val_loss` que controla o early stopping, o checkpoint e a redução da taxa de aprendizado ficaria dominada por poucas imagens.

In [ ]:
MAX_CLASS_WEIGHT = 10.0


def sample_weights_for(df, target, n_classes):
    classes = np.arange(n_classes)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=df[target].values)
    weights = np.minimum(weights, MAX_CLASS_WEIGHT)
    class_to_weight = dict(zip(classes, weights))
    return df[target].map(class_to_weight).astype("float32").values


def make_dataset_with_weights(df, training, batch_size=BATCH_SIZE):
    filenames = df["Image"].values
    labels = (
        df["EXP"].astype("int32").values,
        df["ICM"].astype("int32").values,
        df["TE"].astype("int32").values,
    )
    weights = (
        sample_weights_for(df, "EXP", N_EXP),
        sample_weights_for(df, "ICM", N_ICM),
        sample_weights_for(df, "TE", N_TE),
    )
    ds = tf.data.Dataset.from_tensor_slices((filenames, labels, weights))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    def _map(filename, label, weight):
        img = load_image(filename)
        return img, label, weight

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    if training:
        ds = ds.map(lambda x, y, w: (augmentation(x, training=True), y, w), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds_w = make_dataset_with_weights(train_split_df, training=True)

EPOCHS = 60

history = model.fit(
    train_ds_w,
    validation_data=val_ds,  # sem pesos de classe
    epochs=EPOCHS,
    callbacks=callbacks,
)

# O early stopping só considera epochs a partir de start_from_epoch, mas o checkpoint considera todas.
# Salvar aqui garante que o arquivo seja o mesmo modelo avaliado nas próximas seções.
model.save(CHECKPOINT_PATH)

## 9. Curvas de treino

A loss de treino usa pesos de classe e a de validação não, então compare a tendência de cada curva, não o valor absoluto entre elas.

In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(hist_df["loss"], label="treino", color="#4C72B0")
axes[0, 0].plot(hist_df["val_loss"], label="validação", color="#C44E52")
axes[0, 0].set_title("Loss total")
axes[0, 0].set_xlabel("epoch")
axes[0, 0].legend()

for ax, target in zip(axes.flat[1:], ["exp", "icm", "te"]):
    loss_col = f"{target}_output_loss"
    val_loss_col = f"val_{target}_output_loss"
    ax.plot(hist_df[loss_col], label="treino", color=TARGET_COLORS[target.upper()])
    ax.plot(hist_df[val_loss_col], label="validação", linestyle="--", color=TARGET_COLORS[target.upper()])
    ax.set_title(f"Loss — {target.upper()}")
    ax.set_xlabel("epoch")
    ax.legend()

fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, target in zip(axes, ["exp", "icm", "te"]):
    acc_col = f"{target}_output_accuracy"
    val_acc_col = f"val_{target}_output_accuracy"
    ax.plot(hist_df[acc_col], label="treino", color=TARGET_COLORS[target.upper()])
    ax.plot(hist_df[val_acc_col], label="validação", linestyle="--", color=TARGET_COLORS[target.upper()])
    ax.set_title(f"Accuracy — {target.upper()}")
    ax.set_xlabel("epoch")
    ax.set_ylim(0, 1)
    ax.legend()
fig.tight_layout()
plt.show()

## 10. Avaliação na validação (silver)

In [ ]:
val_results = model.evaluate(val_ds, return_dict=True)
print("Avaliação na validação (silver):")
for k, v in val_results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Métricas por alvo na validação. Em ICM/TE, "avaliáveis" exclui a classe 3 (não determinado),
# como no teste gold, para que a comparação seja justa.
val_preds = model.predict(val_ds.map(lambda x, y: x), verbose=0)

rows = []
for target, probs in zip(["EXP", "ICM", "TE"], val_preds):
    y_true = val_split_df[target].values
    y_pred = probs.argmax(axis=1)
    mask = np.ones(len(y_true), dtype=bool) if target == "EXP" else y_true != 3
    rows.append({
        "alvo": target,
        "n_avaliaveis": int(mask.sum()),
        "accuracy": (y_true[mask] == y_pred[mask]).mean(),
        "linha_de_base": pd.Series(y_true[mask]).value_counts(normalize=True).iloc[0],
        "kappa_quadratico": cohen_kappa_score(y_true[mask], y_pred[mask], weights="quadratic"),
    })
display(pd.DataFrame(rows).round(3))

## 11. Avaliação no conjunto de teste gold

Para cada alvo, linhas com `'ND'` ou `NaN` *naquele alvo* são descartadas apenas da avaliação daquele alvo (a imagem pode permanecer válida para os outros dois alvos).

Como `ND` equivale à classe 3 do treino, essas métricas medem o grau de ICM/TE apenas nas imagens avaliáveis. Se o modelo prevê a classe 3 em uma dessas imagens, isso conta como erro.

In [ ]:
# Predições para todas as imagens do teste gold
test_ds = make_dataset(test_df.assign(EXP=0, ICM=0, TE=0), training=False)  # labels dummy, usados só p/ montar o pipeline
test_images_ds = test_ds.map(lambda x, y: x)
exp_pred, icm_pred, te_pred = model.predict(test_images_ds)

exp_pred_cls = exp_pred.argmax(axis=1)
icm_pred_cls = icm_pred.argmax(axis=1)
te_pred_cls = te_pred.argmax(axis=1)

test_df["EXP_pred"] = exp_pred_cls
test_df["ICM_pred"] = icm_pred_cls
test_df["TE_pred"] = te_pred_cls

In [ ]:
def valid_mask(series):
    return series.notna() & (series.astype(str) != "ND")


def evaluate_target(df, target, n_classes):
    mask = valid_mask(df[target])
    y_true = df.loc[mask, target].astype(int).values
    y_pred = df.loc[mask, f"{target}_pred"].values
    n_excluded = (~mask).sum()

    acc = (y_true == y_pred).mean()
    kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    report = classification_report(y_true, y_pred, labels=np.arange(n_classes), zero_division=0)

    print(f"=== {target} (gold) ===")
    print(f"n válidos: {mask.sum()} | excluídos (ND/NaN): {n_excluded}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Cohen's Kappa (quadrático): {kappa:.4f}")
    print(report)
    return cm, acc, kappa


results = {}
for target, n_classes in [("EXP", N_EXP), ("ICM", N_ICM), ("TE", N_TE)]:
    results[target] = evaluate_target(test_df, target, n_classes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, target in zip(axes, ["EXP", "ICM", "TE"]):
    cm, acc, kappa = results[target]
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{target} (gold) — acc={acc:.2f}, κ={kappa:.2f}")
    ax.set_xlabel("predito")
    ax.set_ylabel("verdadeiro")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)
    ax.set_xticks(range(cm.shape[0]))
    ax.set_yticks(range(cm.shape[0]))
fig.suptitle("Matrizes de confusão — conjunto de teste gold")
fig.tight_layout()
plt.show()

### Embriões inéditos no teste gold

Nenhuma imagem de teste aparece no treino, mas a maioria dos embriões do gold tem outras imagens no CSV de treino silver. Isso pode inflar as métricas gerais. A tabela abaixo separa os embriões que têm imagens no silver dos inéditos, que dão a estimativa mais honesta de generalização. Como o subconjunto inédito é pequeno, seus valores têm alta incerteza.

In [ ]:
silver_embryos = set(train_df["embryo_id"])
is_unseen = ~test_df["embryo_id"].isin(silver_embryos)
print(f"Imagens gold: {len(test_df)} ({test_df['embryo_id'].nunique()} embriões)")
print(f"Inéditas: {is_unseen.sum()} ({test_df.loc[is_unseen, 'embryo_id'].nunique()} embriões)")


def summarize_subset(df, name):
    rows = []
    for target in ["EXP", "ICM", "TE"]:
        mask = valid_mask(df[target])
        y_true = df.loc[mask, target].astype(int).values
        y_pred = df.loc[mask, f"{target}_pred"].values
        rows.append({
            "subconjunto": name,
            "alvo": target,
            "n": int(mask.sum()),
            "accuracy": (y_true == y_pred).mean(),
            "kappa_quadratico": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        })
    return rows


subset_results = pd.DataFrame(
    summarize_subset(test_df, "todos")
    + summarize_subset(test_df[~is_unseen], "embrião presente no silver")
    + summarize_subset(test_df[is_unseen], "embrião inédito")
)
display(subset_results.round(3))

## 12. Conclusões

**Domain shift silver → gold.** O treino usa rótulos de um único embriologista (silver), enquanto o teste usa consenso de um consórcio internacional (gold). Essa diferença de critério de anotação pode reduzir o desempenho no teste em relação à validação (que é amostrada do próprio conjunto silver): o modelo aprende os padrões de um único anotador, não necessariamente o consenso.

**Embriões compartilhados entre treino e teste.** A maioria dos embriões do gold também tem outras imagens no treino silver, o que pode inflar as métricas gerais. A comparação com os embriões inéditos está na seção 11, com a ressalva de que esse subconjunto é pequeno.

**Classe 3 de ICM/TE.** Essa classe marca imagens não avaliáveis (EXP < 2), não um quarto grau de qualidade. Ela é excluída da avaliação no gold, mas o modelo ainda pode prevê-la em imagens avaliáveis, e esse erro aparece nas matrizes de confusão.

**Desbalanceamento de classes.** Classes raras (ex.: EXP=4, ICM=2, TE=2) têm poucas amostras no treino. O `sample_weight` balanceado compensa parte disso, mas não cria exemplos novos. O recall dessas classes deve ser conferido nas matrizes de confusão. O Kappa quadrático penaliza menos erros entre classes vizinhas do que erros distantes, por isso é apropriado para essas escalas ordinais.

**Limitações e próximos passos:**
- *Transfer learning* com um backbone pré-treinado (ex. EfficientNet/ResNet em ImageNet) pode melhorar a extração de features com poucos dados rotulados.
- *Ordinal regression* (ex. CORAL/CORN) respeita a ordem natural de EXP e dos graus de ICM/TE, com a classe 3 tratada à parte.
- *Test-time augmentation* (média de predições sobre várias versões aumentadas da mesma imagem) pode reduzir variância nas predições.
- Agregar múltiplas imagens do mesmo embrião (ensembling por embrião) poderia refletir melhor a prática clínica, em que o embriologista normalmente avalia várias vistas antes de decidir.